<a href="https://colab.research.google.com/github/Aris-1712/LLM-fine-tuning/blob/main/PreferrenceBasedTraining_DPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# install dependencies

!pip install transformers
!pip install datasets
!pip install accelerate
!pip install peft
!pip install trl
!pip install bitsandbytes
!pip install pyMuPdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 21.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 23.2 MB/s eta 0:00:00


In [2]:
# load tokenzier to convert strint to tokens

from transformers import AutoTokenizer
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# tokenizer should be of the model itself
tokenizer = AutoTokenizer.from_pretrained(model)

# set pad token and eos (end of sentence) token
if tokenizer.pad_token == None:
  tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [4]:
# dataset
from datasets import load_dataset

data = load_dataset("json", data_files="/content/DPO_training_dataset.json", split= "train")


Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
data[0]

{'prompt': 'How did Alphabet’s revenue change around Q2 2025 and what does that imply for the business? (case 1)',
 'chosen': 'Alphabet’s revenue shows a year-over-year increase into 2025, reflecting continued strength in its core advertising products alongside growing contributions from cloud services. This combination suggests that demand remains resilient and that the company is successfully diversifying its growth drivers.',
 'rejected': 'Alphabet’s revenue declined into 2025, which implies weakening demand across its products and a loss of traction in both advertising and cloud segments.'}

In [6]:
# Load the Base Model

from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model, device_map="auto")

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [7]:
from peft import LoraConfig, TaskType
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [8]:
# Import exisisting Non Instructional Trained Model

import zipfile

# Open the ZIP file in read mode ('r')
with zipfile.ZipFile('instructional_trained_model.zip', 'r') as zip_ref:
    # Extract all contents to the current directory
    zip_ref.extractall('tinyllama-lora')

In [9]:
# crete the peft model using the lora config

from peft import get_peft_model, PeftModel

path = "/content/tinyllama-lora/instructionalTraining/checkpoint-16"

# Load the base model and non instructional model by merging the weights
InstructionTrainingModel = PeftModel.from_pretrained(model, path)
InstructionTrainingModel = InstructionTrainingModel.merge_and_unload()

peft_model = get_peft_model(InstructionTrainingModel, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [22]:
# DPO training configs

from trl import DPOTrainer, DPOConfig


dpo_args = DPOConfig(
    output_dir="./tinyllama-preference-alignment",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    beta=0.01,
    report_to=[],
    logging_dir=None, # disable logging to wandb or tensorboard
    loss_type="sigmoid",  # or "hinge", depending on experiment
    remove_unused_columns=False
)


trainer = DPOTrainer(
    model=peft_model,
    ref_model=None,
    args=dpo_args,
    train_dataset=data,
    processing_class=tokenizer,   # instead of tokenizer argument
    # you can pass data_collator if needed,
    # optionally eval_dataset etc.
)


trainer.train()


Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Step,Training Loss
10,0.690308
20,0.684875


TrainOutput(global_step=25, training_loss=0.6863053035736084, metrics={'train_runtime': 148.081, 'train_samples_per_second': 1.351, 'train_steps_per_second': 0.169, 'total_flos': 153175835344896.0, 'train_loss': 0.6863053035736084})

In [23]:
from transformers import AutoModelForCausalLM

pathPreference = "/content/tinyllama-preference-alignment/checkpoint-25"

modelPreference = AutoModelForCausalLM.from_pretrained(pathPreference, device_map="auto")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [24]:
prompt_text = "What does the change in Alphabet\u2019s net income indicate about its operational performance? (case 2)"
prompt = f"<s>[INST] {prompt_text} [/INST]"

inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

outputs = modelPreference.generate(
    **inputs,
    max_new_tokens=200, # Increased max_new_tokens
    temperature=0.1,    # Decreased temperature for less randomness
    top_p=0.9,
    do_sample=True,
    repetition_penalty=2.0 # Increased repetition penalty
)


decoded_output = tokenizer.decode(outputs[0],skip_special_tokens=True) # Removed skip_special_tokens=True
print(f"\nDecoded Output (including special tokens):\n{decoded_output}")


Decoded Output (including special tokens):
What does the change in Alphabet’s net income indicate about its operational performance? (case 2)
# - **What is your opinion on this case**: I think that it means a good thing for Google as they are getting more money from advertisers. It also shows how much of their revenue comes through adverts, which can be used to help them with future investments and growth strategies.<br>I would say there're some risks though because if people stop using google products then you might lose out too<jupyter_code><issue_closed>[![Open In Colab](https://colaboratory-badge10x9a6c5843d7fbeeecbceccaeffdfaaedfcbaafef/svg)](http:/google colabs https/)[ ![-F]() ]
